# SMILES augmentation on the ORD-free base — variant 3 re-run on `sagawa/CompoundT5`

Second of the two variants whose recorded failure was explained by a property of the base rather
than by the method (the other is variant 6, notebook 14). `RESULTS.md` on variant 3: augmenting
100% of the time overfit immediately (`eval_loss` rising from the first measurement); dropping to
50% augmentation and `lr` 5e-5 -> 2e-5 removed the overfitting but left accuracy below variant 2
— 36.7% (57k) and 39.3% (147k) against 50.3% ORD top-1.

The mechanism proposed there is that augmentation fought the base's canonical-SMILES prior.
`sagawa/CompoundT5` was span-MLM pretrained on canonical ZINC20 SMILES, so some prior exists —
but it has never generated a reactant set, so the prior is far weaker than ReactionT5's. There is
also a second reason to expect a different outcome here: the 57k CompoundT5 run ended
*undertrained*, not overfit (`eval_loss` fell monotonically to the last step), and augmentation
is a regularizer — the failure mode it caused on ReactionT5 is the one this base is furthest
from.

**Matched control.** Identical to `12_train_reactant_compoundt5_57k.ipynb` — same 57,000-reaction
canonical pool, `lr=5e-4`, 3 epochs, `torchrun --nproc_per_node=2` — with augmentation switched
on as the only difference (drop `--no-augment`; `--augment-prob` defaults to 0.5). Randomization
is applied online to *both* product and reactants, per Tetko et al. (Nature Communications,
2020), and only to the train split — validation stays canonical, so `eval_loss` is directly
comparable to the canonical run's 0.5678.

| | ORD exact top-1 | ORD exact top-5 | ORD core top-5 | eval_loss |
|---|---|---|---|---|
| CompoundT5 + 57k canonical | 17.0% | 23.7% | 34.0% | 0.5678 |
| CompoundT5 + 147k canonical | 19.7% | 28.3% | 39.7% | 0.4185 |
| ReactionT5 variant 2 (57k canonical) | 50.3% | 73.0% | 79.7% | — |
| ReactionT5 variant 3 (57k augmented) | 36.7% | — | — | — |

**Data:** `kuzmenkooleh/retro-planner-reactants-57k-uspto` — the same canonical pool run 12 used;
augmentation happens at training time, not in the dataset.

Training is slower than the canonical run (RDKit re-renders every example on every access), so
the time budget is raised accordingly.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All
(Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

# Kaggle has mounted datasets under two different layouts historically, so search
# rather than hard-code the path.
train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4  # matched to the 57k canonical run, so augmentation is the only difference
augment_prob = 0.5
output_dir = "/kaggle/working/model1_compoundt5_augmented57k"
time_budget_minutes = 260  # canonical 57k took ~89 min; online RDKit re-rendering is slower

In [ ]:
# Must show canonical targets: the root-aligned 57k dataset (notebook 14) carries identical
# filenames, and the glob above would silently accept it, turning this into a rerun of 14.
import json

with open(train_file) as handle:
    for _ in range(3):
        row = json.loads(handle.readline())
        print("product :", row["product_smiles"][:80])
        print("reactant:", row["reactants_smiles"][:80])

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --augment-prob {augment_prob} \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# Augmentation must be ON here (the log line naming augment prob), unlike every other
# CompoundT5 run so far -- and the vocab repair must still have fired.
!grep -E "augmentation ON|new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

# On ReactionT5 this curve turned upward inside the first epoch. Directly comparable to the
# canonical run's 0.5678 because validation is never augmented.
state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[::max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
import os
model_dir = f"{output_dir}/final"
assert os.path.isdir(model_dir), os.listdir(output_dir)

for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/compoundt5_augmented57k_{tag}_topk.json"
    print(tag, "done")

In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/compoundt5_augmented57k_{tag}_topk.json"))
    print("===", tag, "===")
    print(json.dumps(data["summary"], indent=2))